# Property transaction data ingestion

This notebook combines the CSV files in `Data/Raw_data` into the `property_transactions` table in `Output/property_trans.db`. Values are initially ingested as text so identifiers such as postal codes keep their leading zeroes. Column names are normalized to BigQuery-safe snake_case; source CSV files and row values are not modified.

In [3]:
from pathlib import Path

import duckdb
import pandas as pd

In [4]:
# Find the project root whether Jupyter starts in the project or notebook directory.
cwd = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in (cwd, *cwd.parents) if (candidate / "Data" / "Raw_data").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project Data/Raw_data directory from the current working directory.")

raw_data_dir = project_root / "Data" / "Raw_data"
output_dir = project_root / "Output"
database_path = output_dir / "property_trans.db"
csv_files = sorted(raw_data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {raw_data_dir}")

output_dir.mkdir(parents=True, exist_ok=True)
print(f"Project root: {project_root}")
print(f"CSV files to ingest: {len(csv_files)}")
print(f"Database output: {database_path}")

Project root: /home/jenho/Group5_Project2
CSV files to ingest: 19
Database output: /home/jenho/Group5_Project2/Output/property_trans.db


In [5]:
import re

frames = []
encodings = {}
for csv_path in csv_files:
    raw_bytes = csv_path.read_bytes()
    try:
        raw_bytes.decode("utf-8")
        encoding = "utf-8"
    except UnicodeDecodeError:
        encoding = "cp1252"

    frames.append(
        pd.read_csv(csv_path, encoding=encoding, dtype=str, keep_default_na=False)
    )
    encodings[csv_path.name] = encoding

combined_data = pd.concat(frames, ignore_index=True, sort=False).fillna("")
source_column_names = list(combined_data.columns)
normalized_column_names = [
    re.sub(r"[^A-Za-z0-9_]+", "_", name).strip("_").lower()
    for name in source_column_names
]
normalized_column_names = [
    f"_{name}" if name and name[0].isdigit() else name
    for name in normalized_column_names
]
if any(not name for name in normalized_column_names):
    raise ValueError("A source column name could not be normalized to a valid BigQuery field name.")
if len(set(normalized_column_names)) != len(normalized_column_names):
    raise ValueError("Source column names collide after BigQuery-safe normalization.")
combined_data.columns = normalized_column_names

connection = duckdb.connect(str(database_path))
try:
    connection.register("combined_transactions", combined_data)
    connection.execute(
        """
        CREATE OR REPLACE TABLE property_transactions AS
        SELECT * FROM combined_transactions
        """
    )
    row_count = connection.execute("SELECT COUNT(*) FROM property_transactions").fetchone()[0]
    column_names = [row[0] for row in connection.execute("DESCRIBE property_transactions").fetchall()]
finally:
    connection.close()

print(f"Loaded {row_count:,} rows and {len(column_names)} columns from {len(csv_files)} CSV files.")
print(f"Detected encodings: {set(encodings.values())}")
print(f"Created {database_path}")

Loaded 170,224 rows and 21 columns from 19 CSV files.
Detected encodings: {'utf-8', 'cp1252'}
Created /home/jenho/Group5_Project2/Output/property_trans.db


In [6]:
# Reopen the database to verify that the table is persisted.
with duckdb.connect(str(database_path), read_only=True) as connection:
    verification = connection.execute(
        "SELECT COUNT(*) AS row_count FROM property_transactions"
    ).fetchone()[0]

assert verification == row_count, "Persisted row count does not match the ingested row count."
print(f"Verified property_transactions contains {verification:,} rows.")

Verified property_transactions contains 170,224 rows.
